# 06 · File Parsing Pipeline
Multi-format file ingestion: PDF, Python, Markdown, Word docs, Jupyter notebooks.
Domain-aware segmentation producing TextSegment objects for downstream encoding.

**Architecture:** This is Stage 1 of the 5-stage Concept LM pipeline.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.file_parser import FileParser, segments_to_text
import os
parser = FileParser()
print('FileParser loaded. Supported:', parser.SUPPORTED)

## 1. Parse a Markdown file

In [ ]:
sample_md_content = """# Concept-Level Language Models

## Introduction
Large language models apply uniform computation to all tokens.
This is at odds with non-uniform information density of natural language.

## Architecture
The DLCM framework introduces four stages:
encoding, boundary detection, concept reasoning, and decoding.

## Key Insight
Concept boundaries should be learned from data, not imposed by sentence segmenters.
"""

with open('/tmp/sample.md', 'w') as f:
    f.write(sample_md_content)

segs = parser.parse('/tmp/sample.md')
for s in segs:
    print(f"[{s.segment_type:15s}] pos={s.position}: {s.text[:70]!r}")

## 2. Parse a Python file

In [ ]:
sample_py_content = '''
import torch
import torch.nn as nn

class BoundaryDetector(nn.Module):
    def __init__(self, d_token, d_scan):
        super().__init__()
        self.Wq = nn.Linear(d_token, d_scan, bias=False)
        self.Wk = nn.Linear(d_token, d_scan, bias=False)

    def forward(self, H):
        Q = self.Wq(H); K = self.Wk(H)
        cos = (Q[:, :-1] * K[:, 1:]).sum(-1)
        return (1 - cos) / 2


def mean_pool(H, boundaries):
    segments = torch.split(H, boundaries)
    return torch.stack([s.mean(0) for s in segments])
'''

with open('/tmp/sample.py', 'w') as f:
    f.write(sample_py_content)

segs_py = parser.parse('/tmp/sample.py')
for s in segs_py:
    print(f"[{s.segment_type:15s}] pos={s.position}: {s.text[:80]!r}")

## 3. Segments → token sequence

In [ ]:
from transformers import GPT2Tokenizer
tok = GPT2Tokenizer.from_pretrained('gpt2')

segs_md = parser.parse('/tmp/sample.md')
full_text = segments_to_text(segs_md)
input_ids = tok.encode(full_text)

print(f"Segments: {len(segs_md)}")
print(f"Total tokens: {len(input_ids)}")
print(f"Avg tokens/segment: {len(input_ids)/len(segs_md):.1f}")

## 4. Domain segment size distributions

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from transformers import GPT2Tokenizer
tok = GPT2Tokenizer.from_pretrained('gpt2')

segs_md = parser.parse('/tmp/sample.md')
segs_py = parser.parse('/tmp/sample.py')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, segs) in zip(axes, [('markdown', segs_md), ('python', segs_py)]):
    lengths = [len(tok.encode(s.text)) for s in segs if s.text.strip()]
    if not lengths: continue
    ax.hist(lengths, bins=max(3, len(lengths)//2), color='#2E75B6', edgecolor='white')
    ax.axvline(np.mean(lengths), color='red', linestyle='--', label=f'mean={np.mean(lengths):.0f}')
    ax.set_title(f'{name}: segment token lengths')
    ax.set_xlabel('Tokens per segment'); ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout(); plt.show()